# ONLY FOR SHUFFLED DATASET

In [21]:
# ---------------- Imports ----------------
import json
import os
import yaml
from pathlib import Path


In [22]:
# ---------------- Args ----------------
dataset_name = "yield-v1-shuffled-finetuning-min3/test"


In [23]:
# ---------------- Config ----------------
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = os.path.join(config["paths"]["proj_store"])
data_path = os.path.join(proj_store, "data")

INPUT_ROOT = os.path.join(data_path, dataset_name)
OUTPUT_JSON_DIR = os.path.join(proj_store, "evaluation", "generated-utterances-dialogue", f"{dataset_name}", "real")
os.makedirs(OUTPUT_JSON_DIR, exist_ok=True)

OUTPUT_JSON = os.path.join(OUTPUT_JSON_DIR, "dialogue.json")


In [24]:


def convert_dialogue(dialogue):
    turns = []
    turn_id = 0

    for msg in dialogue.get("messages", []):
        if msg.get("role") == "system":
            continue

        turns.append({
            "turn_id": turn_id,
            "timestamp": "",
            "speaker": msg["role"],
            "role": "respondent" if msg["role"] == "user" else "elicitor",
            "utterance": msg["content"]
        })
        turn_id += 1

    return {
        "dialogue_id": dialogue["block_id"],
        "domain": dialogue["domain"],
        "turns": turns
    }

def main():
    all_dialogues = []

    for jsonl_path in Path(INPUT_ROOT).rglob("*.jsonl"):
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                dialogue = json.loads(line)
                all_dialogues.append(convert_dialogue(dialogue))

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_dialogues, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    main()
